# MCDI500 - Fase 4 (Sumativa): Analisis, reproducibilidad y comunicacion de resultados

**Proyecto:** Analisis predictivo de diabetes
**Integrantes:** Daniel Hormazábal, Cristian Pasten, Enso Guidotti
**Curso:** MCDI500 - Programacion para la Ciencia de Datos
**Docente:** Omar Salinas Silva
**Fecha:** 17-06-2026
**Repositorio:** https://github.com/DanielParavel/proyecto-grupo4-mcdi500

---

## Descripcion

Este notebook constituye el **proyecto final integrador (Fase 4)** del proyecto transversal. Integra las cuatro fases del ABP:

- **F1:** Definicion del problema y configuracion del entorno reproducible
- **F2:** Obtencion, limpieza y transformacion del dataset (pipeline funcional)
- **F3:** Nucleo algoritmico POO, recursividad y mediciones de complejidad
- **F4:** Visualizaciones analiticas, storytelling de datos y comunicacion de resultados

El notebook corre de principio a fin con **Kernel -> Restart & Run All** sin errores.

---

## Tabla de contenidos

1. [Importacion y configuracion](#seccion-1)
2. [Carga del dataset RAW (F1-F2)](#seccion-2)
3. [Pipeline POO: preprocesamiento (F2-F3)](#seccion-3)
4. [Validacion tecnica del pipeline (F3)](#seccion-4)
5. [Analisis exploratorio avanzado (F3)](#seccion-5)
6. [Ranking recursivo de variables (F3)](#seccion-6)
7. [Mediciones de complejidad con timeit (F3)](#seccion-7)
8. [Comparacion iterativo vs vectorizado (F3)](#seccion-8)
9. [Storytelling: visualizaciones analiticas (F4)](#seccion-9)
10. [Metodologia, trazabilidad y reflexion (F4)](#seccion-10)
11. [Conclusiones (F4)](#seccion-11)
12. [Bibliografia (APA 7)](#seccion-12)


<a id="seccion-1"></a>

---
## 1. Importacion y configuracion

Stack cientifico del proyecto. Se agrega `seaborn` para las visualizaciones analiticas de F4.


In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import timeit
import tracemalloc
from abc import ABC, abstractmethod

# Agregar src/ al path para importar el modulo reutilizable
_nb_dir = os.getcwd()
_src    = os.path.abspath(os.path.join(_nb_dir, '..', 'src'))
if _src not in sys.path:
    sys.path.insert(0, _src)

# Estilo global coherente
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13, 'axes.titleweight': 'bold'})
plt.rcParams['font.size'] = 11

print(f"pandas     : {pd.__version__}")
print(f"numpy      : {np.__version__}")
print(f"matplotlib : {plt.matplotlib.__version__}")
print(f"seaborn    : {sns.__version__}")
print("\nLibrerias importadas correctamente ✓")


<a id="seccion-2"></a>

---
## 2. Carga del dataset RAW (F1-F2)

Se carga `data/raw/diabetes.csv` (datos crudos del Pima Indians Diabetes Database, version extendida 15.000 registros). La funcion valida la presencia del target y reporta el estado inicial antes del preprocesamiento (McKinney, 2022).


In [ ]:
def cargar_dataset_raw(ruta):
    'Carga el dataset crudo y verifica su estructura basica.'
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No se encontro: {ruta}")
    df = pd.read_csv(ruta)
    assert 'Diabetic' in df.columns, "ERROR: falta la variable objetivo Diabetic"
    print(f"Dataset RAW cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas ✓")
    print(f"Columnas   : {df.columns.tolist()}")
    print(f"Nulos      : {df.isnull().sum().sum()}")
    print(f"Duplicados : {df.duplicated().sum()}")
    print(f"Diabetic   : {df['Diabetic'].value_counts().to_dict()}")
    return df

df_raw = cargar_dataset_raw('../../data/raw/diabetes.csv')

FEATURES = ['Pregnancies', 'PlasmaGlucose', 'DiastolicBloodPressure',
            'TricepsThickness', 'SerumInsulin', 'BMI', 'DiabetesPedigree', 'Age']
TARGET   = 'Diabetic'
print(f"\nFeatures ({len(FEATURES)}): {FEATURES}")


<a id="seccion-3"></a>

---
## 3. Pipeline POO: preprocesamiento (F2-F3)

Las clases del pipeline se importan desde `src/analizador.py`. El pipeline opera sobre los datos crudos demostrando transformaciones efectivas: exclusion de `PatientID`, deduplicacion, imputacion de nulos y normalizacion Min-Max (UNAB, 2026b).

### Arquitectura

```
src/analizador.py
    Transformador (ABC)          <- contrato comun
    +-- ExcluirColumna           <- herencia
    +-- EliminarDuplicados       <- herencia
    +-- ImputarMediana           <- herencia
    +-- NormalizarMinMax         <- herencia
    Pipeline                     <- composicion + polimorfismo
    AnalizadorDiabetes           <- encapsulamiento + cache
```


In [ ]:
from analizador import (
    Transformador, ExcluirColumna, EliminarDuplicados,
    ImputarMediana, NormalizarMinMax, Pipeline, AnalizadorDiabetes,
)

print("Modulo src/analizador.py importado ✓")
print()
for cls in [Transformador, ExcluirColumna, EliminarDuplicados,
            ImputarMediana, NormalizarMinMax, Pipeline, AnalizadorDiabetes]:
    print(f"  ✓ {cls.__name__}")


In [ ]:
pipeline = Pipeline([
    ExcluirColumna('PatientID'),
    EliminarDuplicados(),
    ImputarMediana(),
    NormalizarMinMax(),
])

print("Etapas del pipeline:")
pipeline.listar_etapas()
print()

df = pipeline.ejecutar(df_raw.copy())

print(f"\nDataset procesado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Nulos restantes  : {df.isnull().sum().sum()}")
print(f"Rango features   : [{df[FEATURES].min().min():.4f}, {df[FEATURES].max().max():.4f}]")

# Exportar dataset procesado para trazabilidad F2-F4
ruta_out = '../../data/processed/diabetes_clean.csv'
os.makedirs(os.path.dirname(ruta_out), exist_ok=True)
df.to_csv(ruta_out, index=False)
print(f"\nDataset exportado a {ruta_out} ✓")


<a id="seccion-4"></a>

---
## 4. Validacion tecnica del pipeline (F3)

Cinco pruebas formales cubren los tres escenarios requeridos: casos normales, limites y excepciones.


In [ ]:
print("=" * 60)
print("VALIDACION TECNICA - 5 PRUEBAS FORMALES")
print("=" * 60)

# T1: Normal
print("\n[T1] Normal - Pipeline completo sobre datos RAW")
try:
    df_t = pd.read_csv('../../data/raw/diabetes.csv')
    p_t  = Pipeline([ExcluirColumna('PatientID'), EliminarDuplicados(),
                     ImputarMediana(), NormalizarMinMax()])
    df_o = p_t.ejecutar(df_t)
    assert df_o.isnull().sum().sum() == 0
    assert df_o.shape[0] > 0 and TARGET in df_o.columns
    assert set(df_o[TARGET].unique()) <= {0, 1}
    print("  ✓ 0 nulos, shape intacto, target valido {0,1}")
except Exception as e:
    print(f"  ✗ FALLO: {e}")

# T2: Limite - columna inexistente
print("\n[T2] Limite - ExcluirColumna sobre columna inexistente")
try:
    df_t = df.copy(); shape_antes = df_t.shape
    df_o = Pipeline([ExcluirColumna('ColumnaQueNoExiste')]).ejecutar(df_t)
    assert df_o.shape == shape_antes
    print("  ✓ Shape sin cambios, sin error")
except Exception as e:
    print(f"  ✗ FALLO: {e}")

# T3: Limite - dataset de 1 fila
print("\n[T3] Limite - Pipeline sobre dataset de 1 fila")
try:
    df_o = Pipeline([ExcluirColumna('PatientID'), EliminarDuplicados(),
                     ImputarMediana(), NormalizarMinMax()]).ejecutar(df_raw.iloc[:1].copy())
    assert df_o.shape[0] == 1
    print(f"  ✓ 1 fila procesada correctamente - shape: {df_o.shape}")
except Exception as e:
    print(f"  ✗ FALLO: {e}")

# T4: Limite - columna constante
print("\n[T4] Limite - NormalizarMinMax con columna constante (BMI=0.5)")
try:
    df_t = df.copy(); df_t['BMI'] = 0.5
    df_o = Pipeline([NormalizarMinMax()]).ejecutar(df_t)
    assert df_o.isnull().sum().sum() == 0
    print("  ✓ Sin NaN - columna constante preservada sin division por cero")
except Exception as e:
    print(f"  ✗ FALLO: {e}")

# T5: Excepcion - etapa invalida
print("\n[T5] Excepcion - Pipeline(['string invalido'])")
try:
    Pipeline(["esto_no_es_un_transformador"]).ejecutar(df.copy())
    print("  ✗ FALLO: debia lanzar TypeError")
except TypeError as e:
    print(f"  ✓ TypeError capturado: {e}")

print("\n" + "=" * 60)
print("Validacion completada ✓")


<a id="seccion-5"></a>

---
## 5. Analisis exploratorio avanzado (F3)

La clase `AnalizadorDiabetes` encapsula el analisis con cache interno. Los resultados de esta seccion alimentan directamente las visualizaciones de F4.


In [ ]:
analizador = AnalizadorDiabetes(df, FEATURES, TARGET)

# Estadisticos por clase
stats = analizador.calcular_estadisticos_por_clase()
cols  = ['Media (No diabético)', 'Media (Diabético)', 'Δ Media (1-0)']
print("Estadisticos por clase (ordenado por poder discriminativo):\n")
print(stats[cols].sort_values('Δ Media (1-0)', ascending=False).to_string())
mejor = stats['Δ Media (1-0)'].idxmax()
print(f"\nVariable mas discriminativa: {mejor} (Delta={stats.loc[mejor, 'Δ Media (1-0)']:.4f})")


In [ ]:
# Outliers IQR
outliers = analizador.detectar_outliers_iqr()
print("Deteccion de outliers (IQR):\n")
print(outliers[['N Outliers', '% Outliers', 'Lím. Inf', 'Lím. Sup']]
      .sort_values('N Outliers', ascending=False).to_string())


In [ ]:
# Correlaciones
matriz_corr = analizador.calcular_correlaciones()
corr_target = matriz_corr[TARGET].drop(TARGET).sort_values(ascending=False)
print("Correlacion con Diabetic:\n")
print(corr_target.to_string())


<a id="seccion-6"></a>

---
## 6. Ranking recursivo de variables (F3)

Merge sort recursivo sobre las 8 features por correlacion absoluta. Complejidad O(p*log p).


In [ ]:
ranking = analizador.rankear_variables_recursivo()
print("Ranking (merge sort recursivo):\n")
print(ranking.to_string())


<a id="seccion-7"></a>

---
## 7. Mediciones de complejidad con timeit (F3)

Se mide el tiempo de cada metodo sobre 5 tamaños de dataset (1k a 15k filas), verificando empiricamente las complejidades teoricas.


In [ ]:
def medir_con_timeit(df, features, target, tamanos=None, n_rep=3):
    'Mide el tiempo de cada metodo del AnalizadorDiabetes con timeit.'
    if tamanos is None:
        tamanos = [1000, 3000, 6000, 10000, 15000]
    registros = []
    for n in tamanos:
        df_sub  = df.iloc[:n].copy()
        t_stats = timeit.timeit(lambda: AnalizadorDiabetes(df_sub, features, target).calcular_estadisticos_por_clase(), number=n_rep) / n_rep
        t_iqr   = timeit.timeit(lambda: AnalizadorDiabetes(df_sub, features, target).detectar_outliers_iqr(), number=n_rep) / n_rep
        t_corr  = timeit.timeit(lambda: AnalizadorDiabetes(df_sub, features, target).calcular_correlaciones(), number=n_rep) / n_rep
        az_temp = AnalizadorDiabetes(df_sub, features, target)
        az_temp.calcular_correlaciones()
        t_rank  = timeit.timeit(lambda: az_temp.rankear_variables_recursivo(), number=n_rep) / n_rep
        registros.append({'N': n,
            'Stats O(n*p) ms':    round(t_stats * 1000, 2),
            'IQR O(n*p) ms':      round(t_iqr   * 1000, 2),
            'Corr O(n*p2) ms':    round(t_corr  * 1000, 2),
            'Rank O(p*logp) ms':  round(t_rank  * 1000, 3),
        })
        print(f"  n={n:<6} stats={t_stats*1000:.1f}ms  iqr={t_iqr*1000:.1f}ms  corr={t_corr*1000:.1f}ms  rank={t_rank*1000:.2f}ms")
    return pd.DataFrame(registros).set_index('N')

print("Midiendo complejidad temporal (timeit, n_rep=3)...\n")
df_tiempos = medir_con_timeit(df, FEATURES, TARGET)
print("\nTabla de tiempos (ms):")
print(df_tiempos.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colores_map = {
    'Stats O(n*p) ms':   'steelblue',
    'IQR O(n*p) ms':     'seagreen',
    'Corr O(n*p2) ms':   'coral',
    'Rank O(p*logp) ms': 'mediumpurple',
}

for col, color in colores_map.items():
    axes[0].plot(df_tiempos.index, df_tiempos[col], 'o-', label=col, color=color, linewidth=2)
axes[0].set_xlabel('Numero de filas')
axes[0].set_ylabel('Tiempo (ms)')
axes[0].set_title('Escalamiento temporal por metodo')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Solo los que crecen con n para ver mejor la diferencia
for col in ['Stats O(n*p) ms', 'Corr O(n*p2) ms']:
    axes[1].plot(df_tiempos.index, df_tiempos[col], 'o-',
                 label=col, color=colores_map[col], linewidth=2)
axes[1].set_xlabel('Numero de filas')
axes[1].set_ylabel('Tiempo (ms)')
axes[1].set_title('O(n*p) vs O(n*p2) - diferencia de crecimiento')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


<a id="seccion-8"></a>

---
## 8. Comparacion iterativo vs vectorizado (F3)

Dos implementaciones del calculo de media. El speedup cuantifica la ventaja de delegar a NumPy (C compilado) frente al overhead del interprete Python (VanderPlas, 2023).


In [ ]:
def calcular_media_iterativo(df, col):
    'Calcula la media iterando fila por fila. O(n) con overhead Python.'
    total = 0.0
    n = 0
    for valor in df[col]:
        total += valor
        n += 1
    return total / n if n > 0 else 0.0

def calcular_media_vectorizado(df, col):
    'Calcula la media con operacion vectorizada. O(n) delegado a C.'
    return df[col].mean()

col_test = 'PlasmaGlucose'
n_reps   = 50

t_iter = timeit.timeit(lambda: calcular_media_iterativo(df, col_test),  number=n_reps) / n_reps
t_vect = timeit.timeit(lambda: calcular_media_vectorizado(df, col_test), number=n_reps) / n_reps

r_iter = calcular_media_iterativo(df, col_test)
r_vect = calcular_media_vectorizado(df, col_test)
assert abs(r_iter - r_vect) < 1e-10, "ERROR: resultados distintos"

speedup = t_iter / t_vect
print(f"Iterativo  (iterrows): {t_iter*1000:.2f} ms")
print(f"Vectorizado (pandas) : {t_vect*1000:.4f} ms")
print(f"Speedup              : {speedup:.0f}x mas rapido (vectorizado)")
print(f"Resultados equivalentes: diferencia = {abs(r_iter - r_vect):.2e} ✓")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['Iterativo\n(Python puro)', f'Vectorizado\n(pandas/NumPy)'],
       [t_iter * 1000, t_vect * 1000], color=['coral', 'steelblue'], width=0.5)
ax.set_ylabel('Tiempo promedio (ms)')
ax.set_title(f'Iterativo vs Vectorizado - Speedup: {speedup:.0f}x')
for i, v in enumerate([t_iter * 1000, t_vect * 1000]):
    ax.text(i, v + 0.5, f'{v:.2f} ms', ha='center', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


<a id="seccion-9"></a>

---
## 9. Storytelling: visualizaciones analiticas (F4)

Los tres graficos cuentan una historia progresiva sobre los factores asociados a la diabetes tipo 2 en el dataset:

- **Acto 1 - Contexto:** Distribucion de la variable objetivo. Cuantos pacientes son diabeticos y cuantos no?
- **Acto 2 - Conflicto:** Las mujeres con mas embarazos tienen mayor prevalencia de diabetes. Pregnancies es la variable mas discriminativa.
- **Acto 3 - Resolucion:** El mapa de correlaciones revela que Pregnancies, Age y PlasmaGlucose son las variables mas relacionadas con el diagnostico.


### Acto 1 - Contexto: distribucion de la variable objetivo

**Pregunta:** Como se distribuyen los diagnosticos en el dataset?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Conteos absolutos
conteos = df[TARGET].value_counts().sort_index()
etiquetas = ['No diabetico (0)', 'Diabetico (1)']
colores   = ['steelblue', 'coral']

axes[0].bar(etiquetas, conteos.values, color=colores, edgecolor='white', linewidth=1.2)
axes[0].set_title('El dataset contiene un 33% de casos diabeticos\n(balance aceptable para analisis exploratorio)')
axes[0].set_ylabel('Numero de pacientes')
axes[0].set_xlabel('Diagnostico')
for i, v in enumerate(conteos.values):
    axes[0].text(i, v + 100, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold', fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim(0, conteos.max() * 1.2)

# Proporcion en pie
axes[1].pie(conteos.values, labels=etiquetas, colors=colores, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporcion diagnostico\n(67% no diabetico, 33% diabetico)')

plt.tight_layout()
plt.savefig('viz_acto1_distribucion_objetivo.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nQue muestra: el dataset tiene 15.000 registros con un balance 67/33.")
print("Que infiero: el dataset no tiene desbalance severo, lo que facilita el analisis exploratorio.")
print("Como lo cuento: casi 1 de cada 3 pacientes tiene diagnostico positivo, lo que representa una prevalencia alta respecto a la poblacion general.")


### Acto 2 - Conflicto: Pregnancies como variable mas discriminativa

**Pregunta:** Por que Pregnancies es la variable con mayor diferencia de medias entre clases?


In [ ]:
# Obtener top 4 features por poder discriminativo
top4 = stats['Δ Media (1-0)'].sort_values(ascending=False).head(4).index.tolist()

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

colores_clase = {0: 'steelblue', 1: 'coral'}
etiquetas_clase = {0: 'No diabetico', 1: 'Diabetico'}

for i, feature in enumerate(top4):
    for clase in [0, 1]:
        datos = df[df[TARGET] == clase][feature]
        axes[i].hist(datos, bins=30, alpha=0.6,
                     color=colores_clase[clase],
                     label=etiquetas_clase[clase],
                     edgecolor='white')
    # Lineas de media por clase
    for clase in [0, 1]:
        media = df[df[TARGET] == clase][feature].mean()
        axes[i].axvline(media, color=colores_clase[clase],
                       linestyle='--', linewidth=2, alpha=0.9)

    delta = stats.loc[feature, 'Δ Media (1-0)']
    axes[i].set_title(f'{feature}\n(Delta media = {delta:.3f})', fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Valor normalizado [0,1]')
    axes[i].legend(fontsize=8)
    axes[i].grid(True, alpha=0.3)

axes[0].set_ylabel('Frecuencia')
plt.suptitle('Las 4 variables mas discriminativas separan claramente ambas clases\n(lineas punteadas = media por clase)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('viz_acto2_distribuciones_discriminativas.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nQue muestra: los histogramas superpuestos revelan que {top4[0]} tiene la mayor separacion entre clases (Delta={stats.loc[top4[0], chr(0x394)+' Media (1-0)']:.3f}).")
print("Que infiero: a mayor numero de embarazos, mayor probabilidad de diagnostico diabetico.")
print("Como lo cuento: las exposiciones hormonales repetidas en embarazos sucesivos deterioran las celulas beta pancreaticas, elevando el riesgo de diabetes tipo 2.")


### Acto 3 - Resolucion: mapa de correlaciones

**Pregunta:** Que variables tienen la relacion mas directa con el diagnostico?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap completo
im = axes[0].imshow(matriz_corr.values, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=axes[0])
axes[0].set_xticks(range(len(matriz_corr.columns)))
axes[0].set_yticks(range(len(matriz_corr.columns)))
axes[0].set_xticklabels(matriz_corr.columns, rotation=45, ha='right', fontsize=8)
axes[0].set_yticklabels(matriz_corr.columns, fontsize=8)
for i in range(len(matriz_corr)):
    for j in range(len(matriz_corr.columns)):
        val = matriz_corr.values[i, j]
        color_texto = 'white' if abs(val) > 0.5 else 'black'
        axes[0].text(j, i, f"{val:.2f}", ha='center', va='center',
                     fontsize=7, color=color_texto)
axes[0].set_title('Matriz de correlacion de Pearson\n(todas las variables)')

# Barras de correlacion con Diabetic - destacando las significativas
corr_ord = matriz_corr[TARGET].drop(TARGET).sort_values(ascending=True)
colores_barra = ['coral' if v >= 0.3 else 'steelblue' if v >= 0.1 else 'lightgray'
                 for v in corr_ord.values]
bars = axes[1].barh(corr_ord.index, corr_ord.values, color=colores_barra, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].axvline(0.3, color='coral', linestyle='--', linewidth=1.5, alpha=0.7, label='Correlacion alta (>0.3)')
axes[1].axvline(0.1, color='steelblue', linestyle='--', linewidth=1.5, alpha=0.7, label='Correlacion moderada (>0.1)')
for bar, val in zip(bars, corr_ord.values):
    axes[1].text(val + 0.005 if val >= 0 else val - 0.005, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
axes[1].set_xlabel('Correlacion de Pearson con Diabetic')
axes[1].set_title('Pregnancies, Age y PlasmaGlucose son\nlas variables mas relacionadas con el diagnostico')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('viz_acto3_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nQue muestra: Pregnancies (r=0.41), Age y PlasmaGlucose tienen las correlaciones positivas mas altas con el diagnostico.")
print("Que infiero: estas tres variables son las mejores candidatas como features principales para el modelado en F4.")
print("Como lo cuento: el mapa de correlaciones confirma el hallazgo del ranking recursivo de F3 y orienta la seleccion de features para el modelo predictivo.")


<a id="seccion-10"></a>

---
## 10. Metodologia, trazabilidad y reflexion (F4)

### Tabla comparativa F1-F4


In [ ]:
trazabilidad = {
    'Fase': ['F1', 'F2', 'F3', 'F4'],
    'Descripcion': [
        'Definicion del problema y configuracion del entorno',
        'Obtencion, limpieza y transformacion del dataset',
        'Nucleo algoritmico POO, recursividad y complejidad',
        'Visualizaciones, storytelling y comunicacion de resultados',
    ],
    'Herramientas': [
        'Python 3.11, venv, Git, Jupyter',
        'pandas, numpy, funciones de preprocesamiento',
        'ABC, Pipeline POO, merge sort, timeit, tracemalloc',
        'matplotlib, seaborn, src/analizador.py',
    ],
    'Mejoras aplicadas': [
        'Entorno reproducible con requirements.txt',
        'Pipeline refactorizado a clases (F3)',
        'Pipeline sobre RAW, modulo src/, visualizaciones (F4)',
        'Storytelling con 3 actos, exportacion a PNG, changelog.md',
    ],
    'Estado': ['✓ Completada', '✓ Completada', '✓ Completada', '✓ Completada'],
}

df_traza = pd.DataFrame(trazabilidad).set_index('Fase')
print("TRAZABILIDAD F1-F4\n")
print(df_traza.to_string())


In [ ]:
# Complejidad espacial con tracemalloc
print("COMPLEJIDAD ESPACIAL (tracemalloc)\n")

def medir_memoria(func):
    'Mide memoria pico (KB) con tracemalloc.'
    tracemalloc.start()
    resultado = func()
    _, pico   = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return resultado, pico / 1024

az_mem = AnalizadorDiabetes(df, FEATURES, TARGET)
metodos = [
    ('calcular_estadisticos_por_clase', az_mem.calcular_estadisticos_por_clase),
    ('detectar_outliers_iqr',           az_mem.detectar_outliers_iqr),
    ('calcular_correlaciones',          az_mem.calcular_correlaciones),
    ('rankear_variables_recursivo',     az_mem.rankear_variables_recursivo),
]

memorias = {}
print(f"{'Metodo':<40} {'Memoria pico (KB)':>18}")
print("-" * 60)
for nombre, func in metodos:
    _, mem_kb = medir_memoria(func)
    memorias[nombre] = mem_kb
    print(f"{nombre:<40} {mem_kb:>17.1f}")
print("\n✓ Medicion espacial completada")


<a id="seccion-11"></a>

---
## 11. Conclusiones (F4)

### Hallazgos principales

1. **Variable mas discriminativa:** Pregnancies presenta la mayor diferencia de medias entre clases (Delta=0.41), con sustento clinico: exposiciones hormonales repetidas deterioran las celulas beta pancreaticas.

2. **Correlaciones con el diagnostico:** Pregnancies (r=0.41), Age y PlasmaGlucose son las tres variables mas relacionadas con Diabetic, lo que orienta la seleccion de features para el modelado en F5.

3. **Eficiencia algoritmica:** La implementacion vectorizada es aproximadamente 41x mas rapida que la iterativa. El merge sort recursivo opera en O(p*log p) y escala mejor que un ordenamiento cuadratico.

4. **Modularidad:** La refactorizacion a `src/analizador.py` separa la logica de negocio de la presentacion y hace el codigo reutilizable en cualquier notebook del proyecto.

### Limitaciones

- La correlacion de Pearson asume linealidad; relaciones no lineales entre variables y el diagnostico no quedan capturadas.
- El dataset no incluye variables temporales ni de seguimiento clinico, lo que limita inferencias causales.
- La normalizacion Min-Max es sensible a outliers extremos; una normalizacion robusta podria mejorar la calidad para modelos de distancia.

### Proyeccion a F5

- Entrenar modelos clasificadores (Logistic Regression, Random Forest, SVM) sobre el dataset procesado.
- Evaluar con metricas adecuadas para clasificacion binaria: precision, recall, F1-score, ROC-AUC.
- Incorporar seleccion de features basada en el ranking de correlaciones de F3.


In [ ]:
print("RESUMEN DEL NOTEBOOK - FASE 4\n")
resumen = {
    'Componente': [
        'Carga del dataset RAW (F1-F2)',
        'Pipeline POO desde src/analizador.py (F2-F3)',
        'Exportacion diabetes_clean.csv (F2)',
        'Validacion tecnica - 5 pruebas (F3)',
        'AnalizadorDiabetes - estadisticos, outliers, correlaciones (F3)',
        'Ranking recursivo merge sort - O(p*log p) (F3)',
        'Mediciones timeit - 5 tamanios (F3)',
        'Comparacion iterativo vs vectorizado - ~41x (F3)',
        'Acto 1 - Distribucion variable objetivo (F4)',
        'Acto 2 - Top 4 features discriminativas (F4)',
        'Acto 3 - Mapa de correlaciones (F4)',
        'Trazabilidad F1-F4 (F4)',
        'Complejidad espacial tracemalloc (F3-F4)',
    ],
    'Estado': ['✓'] * 13,
}
df_res = pd.DataFrame(resumen)
print(df_res.to_string(index=False))
print("\n✓ Fase 4 completada - notebook ejecutado sin errores")


<a id="seccion-12"></a>

---
## 12. Bibliografia (APA 7)

**Material docente**

UNAB. (2026a). *Programa de asignatura MCDI500 - Programacion para la Ciencia de Datos*. Universidad Andres Bello, Facultad de Ingenieria.

UNAB. (2026b). *Guia de desarrollo - Sumativa 4 (Fase 4)*. MCDI500, Universidad Andres Bello.

**Documentacion oficial**

McKinney, W. (2022). *Python for Data Analysis: Data Wrangling with pandas, NumPy, and Jupyter* (3.a ed.). O'Reilly Media. https://wesmckinney.com/book/

The pandas development team. (2024). *pandas documentation (v2.2.2)*. https://pandas.pydata.org/docs/

Waskom, M. (2021). Seaborn: statistical data visualization. *Journal of Open Source Software, 6*(60), 3021. https://doi.org/10.21105/joss.03021

**Bibliografia complementaria**

Ramalho, L. (2022). *Fluent Python: Clear, Concise, and Effective Programming* (2.a ed.). O'Reilly Media.

VanderPlas, J. (2023). *Python Data Science Handbook: Essential Tools for Working with Data* (2.a ed.). O'Reilly Media. https://jakevdp.github.io/PythonDataScienceHandbook/
